# TextCNN para análise de sentimentos (positivo × negativo)

Exemplo prático e simples da **TextCNN** de Yoon Kim (2014), *Convolutional Neural Networks for Sentence Classification*, implementada em PyTorch.

**Tarefa:** ler uma crítica de filme em inglês e dizer se ela é **positiva** ou **negativa**.

**Roteiro do notebook:**

1. instalar e importar as bibliotecas;
2. baixar o dataset IMDb pela biblioteca `datasets` (sem cadastro nem chave) e separar um subconjunto pequeno e balanceado;
3. pré-processar: tokenizar, montar o vocabulário e aplicar *padding*;
4. montar a TextCNN: *embedding* → 3 convoluções 1D (n-gramas de 3, 4 e 5 palavras) → *max-pooling* no tempo → concatenação → *dropout* → camada linear;
5. treinar por 5 épocas e acompanhar perda e acurácia;
6. testar frases novas com `prever_sentimento(frase)`.

Para rodar rápido em aula, usamos só 4.000 críticas de treino e 1.000 de teste. Em CPU, o notebook todo leva por volta de 1 a 2 minutos.

In [ ]:
# Instala as bibliotecas no mesmo Python do kernel (no Colab, torch e matplotlib já vêm instalados).
# Depois de instalar pela primeira vez, reinicie o kernel.
%pip install -q datasets torch matplotlib ipywidgets

In [ ]:
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import concatenate_datasets, load_dataset
from torch.utils.data import DataLoader, TensorDataset

# Semente fixa: o mesmo resultado a cada execução
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Rodando em:", device)

## 1. Os dados: críticas do IMDb

O dataset `stanfordnlp/imdb` tem 50 mil críticas de filmes em inglês, já rotuladas:

- `label = 0` → **negativa**
- `label = 1` → **positiva**

A função `load_dataset` baixa tudo direto da Hugging Face, sem login. Para o treino ser rápido, montamos um **subconjunto balanceado**: metade negativas e metade positivas, escolhidas ao acaso com `seed=42`.

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)


def amostra_balanceada(parte, n):
    # Embaralha e pega n/2 críticas negativas e n/2 positivas
    parte = parte.shuffle(seed=SEED)
    neg = parte.filter(lambda ex: ex["label"] == 0).select(range(n // 2))
    pos = parte.filter(lambda ex: ex["label"] == 1).select(range(n // 2))
    return concatenate_datasets([neg, pos]).shuffle(seed=SEED)


treino = amostra_balanceada(imdb["train"], 4000)
teste = amostra_balanceada(imdb["test"], 1000)

print(f"\nTreino: {len(treino)} críticas | positivas: {sum(treino['label'])}")
print(f"Teste:  {len(teste)} críticas | positivas: {sum(teste['label'])}")
print("\nExemplo:", treino[0]["text"][:300], "...")
print("Rótulo:", treino[0]["label"])

## 2. Pré-processamento

A rede não lê texto, só números. Por isso cada crítica passa por três passos:

1. **Tokenização:** minúsculas, remoção do HTML `<br />` e da pontuação, e `split()` para separar as palavras.
2. **Vocabulário (`word2idx`):** cada palavra recebe um número. Guardamos só as **10.000 palavras mais frequentes** do treino, mais dois tokens especiais:
   - `<pad>` (índice 0): "enchimento", usado para completar críticas curtas;
   - `<unk>` (índice 1): qualquer palavra fora do vocabulário.
3. **Tamanho fixo (`max_len = 200`):** críticas mais longas são **cortadas** e as mais curtas são **completadas com `<pad>`**. Assim todas as críticas de um lote viram uma matriz retangular, que é o que a rede precisa.

In [ ]:
def tokenizar(texto):
    texto = texto.lower().replace("<br />", " ")   # minúsculas e sem quebra de linha em HTML
    texto = re.sub(r"[^a-z0-9' ]", " ", texto)      # troca pontuação por espaço
    return texto.split()


print(tokenizar("This movie was GREAT!<br /><br />I'd watch it again..."))

In [ ]:
MAX_VOCAB = 10_000

contagem = Counter(palavra for texto in treino["text"] for palavra in tokenizar(texto))

word2idx = {"<pad>": 0, "<unk>": 1}
for palavra, _ in contagem.most_common(MAX_VOCAB):
    word2idx[palavra] = len(word2idx)

print("Tamanho do vocabulário:", len(word2idx))
print("Palavras mais frequentes:", list(word2idx)[2:22])

In [ ]:
MAX_LEN = 200


def codificar(texto, max_len=MAX_LEN):
    ids = [word2idx.get(p, 1) for p in tokenizar(texto)]   # 1 = <unk>
    ids = ids[:max_len]                                      # trunca as longas
    ids = ids + [0] * (max_len - len(ids))                   # completa as curtas com <pad> (0)
    return ids


# Exemplo: uma frase curta vira 200 números, quase todos 0 (<pad>)
exemplo = codificar("I loved this movie, the acting was brilliant")
print(exemplo[:15], "... (total:", len(exemplo), "números)")

In [ ]:
def montar_tensores(parte):
    X = torch.tensor([codificar(t) for t in parte["text"]], dtype=torch.long)
    y = torch.tensor(parte["label"], dtype=torch.float)   # float, por causa da BCEWithLogitsLoss
    return X, y


X_treino, y_treino = montar_tensores(treino)
X_teste, y_teste = montar_tensores(teste)

loader_treino = DataLoader(TensorDataset(X_treino, y_treino), batch_size=64, shuffle=True)
loader_teste = DataLoader(TensorDataset(X_teste, y_teste), batch_size=256)

print("X_treino:", tuple(X_treino.shape), "→ (críticas, palavras)")

## 3. A arquitetura TextCNN

### 3.1 Embedding: a crítica vira uma matriz

A camada `nn.Embedding` troca cada índice por um vetor de `EMBED_DIM = 100` números, aprendido durante o treino. Uma crítica de 200 palavras vira uma **matriz 200 × 100**: uma linha por palavra. Com `padding_idx=0`, o vetor do `<pad>` fica fixo em zero e não atrapalha.

### 3.2 Convolução 1D: filtros que deslizam sobre as palavras

Cada filtro tem **largura k** (3, 4 ou 5 palavras) e ocupa **todas as 100 dimensões** do embedding. Ele desliza **só ao longo do tempo** (de palavra em palavra), por isso a convolução é 1D:

```
palavras →   the   movie   was   really   great   and   ...
             [====== filtro k=3 =====]
                    [====== filtro k=3 =====]
                            [====== filtro k=3 ======]  ...
```

Em cada posição, o filtro faz um produto escalar com o trecho de k palavras e devolve **um número**: quão bem aquele trecho combina com o padrão que ele aprendeu (por exemplo, "was really great" ou "waste of time"). Na prática, **cada filtro é um detector de n-gramas**.

No PyTorch, `nn.Conv1d` espera a entrada no formato `(lote, canais, tempo)`. As 100 dimensões do embedding são os **canais**, e as palavras são o **tempo**. Por isso trocamos os eixos com `permute(0, 2, 1)`.

Usamos 3 convoluções em paralelo, com `kernel_size` = 3, 4 e 5 e **100 filtros cada**.

### 3.3 ReLU e max-pooling ao longo do tempo

Depois da ReLU, cada filtro produz uma sequência de valores, um por posição. O **max-pooling global no tempo** guarda só o **maior** deles: "esse padrão apareceu em algum lugar da crítica, e com que força?". A posição exata é descartada.

Vantagem: qualquer que seja o tamanho da frase, cada filtro devolve **um único número**.

### 3.4 Concatenação, dropout e saída

Cada convolução dá 100 números (um por filtro). **Concatenamos** os três vetores para juntar os detectores de 3, 4 e 5 palavras num só resumo da crítica, com **3 × 100 = 300** números. Daí vêm:

- `Dropout(0.5)`: no treino, zera metade dos valores ao acaso, para a rede não decorar o treino;
- `Linear(300, 1)`: um único número (o *logit*). Passando pela sigmoide, ele vira a **probabilidade de a crítica ser positiva**.

```
crítica (200 palavras)
   → Embedding               (200 × 100)
   → Conv1d k=3 / k=4 / k=5  (100 filtros cada) + ReLU
   → Max-pooling no tempo    (100 + 100 + 100)
   → Concatenação            (300)
   → Dropout(0.5) → Linear(300, 1) → sigmoide → P(positivo)
```

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, kernel_sizes=(3, 4, 5), num_filters=100, dropout=0.5):
        super().__init__()
        # Cada palavra vira um vetor de embed_dim números; o <pad> (índice 0) fica sempre em zero
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Uma convolução 1D para cada tamanho de n-grama (3, 4 e 5 palavras)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)   # 3 × 100 = 300 → 1

    def forward(self, x):
        # x: (lote, max_len) com os índices das palavras
        emb = self.embedding(x)              # (lote, max_len, embed_dim)
        emb = emb.permute(0, 2, 1)           # (lote, embed_dim, max_len): formato do Conv1d

        resumos = []
        for conv in self.convs:
            c = torch.relu(conv(emb))        # (lote, num_filters, max_len - k + 1)
            m = c.max(dim=2).values          # max-pooling no tempo → (lote, num_filters)
            resumos.append(m)

        h = torch.cat(resumos, dim=1)        # concatena → (lote, 300)
        h = self.dropout(h)
        return self.fc(h).squeeze(1)         # (lote,): um logit por crítica


modelo = TextCNN(vocab_size=len(word2idx)).to(device)
print(modelo)
print("Parâmetros treináveis:", sum(p.numel() for p in modelo.parameters() if p.requires_grad))

## 4. Treinamento

- **Perda:** `BCEWithLogitsLoss`, a entropia cruzada binária. Ela já aplica a sigmoide internamente, por isso a rede devolve o *logit* "cru".
- **Otimizador:** Adam, com taxa de aprendizado 0,001.
- **Previsão:** logit > 0 (ou seja, probabilidade > 0,5) → positiva.

A cada época, medimos a perda e a acurácia no treino e no teste.

In [ ]:
criterio = nn.BCEWithLogitsLoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


def avaliar(loader):
    # Perda média e acurácia, sem atualizar os pesos
    modelo.eval()
    perda_total, acertos, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = modelo(X)
            perda_total += criterio(logits, y).item() * len(y)
            acertos += ((logits > 0).float() == y).sum().item()
            total += len(y)
    return perda_total / total, acertos / total


def treinar_uma_epoca(loader):
    modelo.train()
    perda_total, acertos, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = modelo(X)                 # 1. passo para frente
        perda = criterio(logits, y)        # 2. calcula a perda
        otimizador.zero_grad()
        perda.backward()                   # 3. gradientes
        otimizador.step()                  # 4. atualiza os pesos
        perda_total += perda.item() * len(y)
        acertos += ((logits > 0).float() == y).sum().item()
        total += len(y)
    return perda_total / total, acertos / total

In [ ]:
EPOCAS = 5
historico = {"perda_treino": [], "perda_teste": [], "acc_treino": [], "acc_teste": []}

for epoca in range(1, EPOCAS + 1):
    perda_tr, acc_tr = treinar_uma_epoca(loader_treino)
    perda_te, acc_te = avaliar(loader_teste)

    historico["perda_treino"].append(perda_tr)
    historico["perda_teste"].append(perda_te)
    historico["acc_treino"].append(acc_tr)
    historico["acc_teste"].append(acc_te)

    print(f"Época {epoca}/{EPOCAS} | perda treino {perda_tr:.3f}  acc treino {acc_tr:.1%} | "
          f"perda teste {perda_te:.3f}  acc teste {acc_te:.1%}")

## 5. Evolução do treino

À esquerda, a perda; à direita, a acurácia. Se a curva de treino continua melhorando enquanto a de teste para ou piora, o modelo começou a **decorar** o treino (*overfitting*), o que é comum com só 4.000 críticas.

In [ ]:
epocas = range(1, EPOCAS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(epocas, historico["perda_treino"], "o-", label="treino")
ax1.plot(epocas, historico["perda_teste"], "s-", label="teste")
ax1.set_title("Perda (BCE)")
ax1.set_xlabel("época")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epocas, historico["acc_treino"], "o-", label="treino")
ax2.plot(epocas, historico["acc_teste"], "s-", label="teste")
ax2.set_title("Acurácia")
ax2.set_xlabel("época")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Testando frases novas

A função `prever_sentimento(frase)` faz com uma frase nova o mesmo caminho do treino: tokeniza, converte em índices, aplica o *padding*, passa pela rede e transforma o logit em probabilidade com a sigmoide.

As frases precisam estar **em inglês**, a língua do IMDb.

In [ ]:
def prever_sentimento(frase):
    modelo.eval()
    x = torch.tensor([codificar(frase)], dtype=torch.long).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(modelo(x)).item()   # probabilidade de ser positiva
    classe = "POSITIVO" if prob >= 0.5 else "NEGATIVO"
    print(f"{classe:8} | P(positivo) = {prob:.2f} | {frase}")
    return prob, classe


frases = [
    "This movie is great and the acting was amazing",
    "Terrible plot, boring and a complete waste of time",
    "One of the best films I have ever seen, a true masterpiece",
    "The worst movie of the year, awful acting and a stupid story",
    "Not bad at all, I actually enjoyed it",                              # negação: caso difícil
    "Beautiful visuals, but the story was boring and the ending was weak",  # opinião mista
]

for f in frases:
    prever_sentimento(f)

Teste as suas próprias frases:

In [ ]:
prever_sentimento("I really loved the soundtrack and the characters")

## 7. Limitações e próximos passos

- **Contexto curto:** cada filtro enxerga no máximo 5 palavras seguidas, e o max-pooling descarta a posição. Negações ("not bad"), opiniões mistas e ironia costumam enganar o modelo.
- **Poucos dados:** usamos só 4.000 críticas. Com as 25 mil do treino completo, a acurácia no teste sobe.
- **Embeddings do zero:** os vetores das palavras são aprendidos só com essas críticas. Começar de vetores pré-treinados, como o **GloVe**, costuma melhorar o resultado (é o que o D2L faz, na seção 16.3).
- **Modelos contextuais**, como o BERT, lidam melhor com negação e ironia, a um custo computacional bem maior.